# S6E9 · V9.1 — RealMLP-style PyTorch · Kaggle GPU

这版专门为 **Kaggle GPU** 重做，不再使用你 Mac 上非常慢的 PyTabKit CPU 路线。

目标：

- 原生 PyTorch + CUDA
- 默认 **5 folds × 4 epochs**
- 自动支持单 GPU / Kaggle T4×2 (`DataParallel`)
- AMP mixed precision
- 每折输出 AUC
- 保存完整 OOF
- 保存 test prediction
- 可选加入原始 `EV_Adoption_and_Range_Anxiety_Dataset.csv`
- 最终产生可以拿回本地加入 V8 Nested Hill 的两个关键文件

输出：

```text
/kaggle/working/oof_realmlp_v91.csv
/kaggle/working/submission_realmlp_v91.csv
/kaggle/working/fold_scores_realmlp_v91.csv
/kaggle/working/realmlp_v91_report.json
```

## 为什么不是上一版的 256 epochs？

公开的 S6E9 RealMLP PyTorch notebook 本身是 GPU notebook，页面显示约 10 分钟运行完成。
这一版采用短 epoch、大 batch、GPU mixed precision 的路线。

> 说明：这是根据公开 notebook 可见的训练结构和 RealMLP 思路做的独立 GPU 实现，
> 不是对作者私有/未暴露代码的逐行复制。

## 0. Kaggle 上先这样设置

1. 打开 Kaggle → **New Notebook**
2. 右侧 **Settings**
3. Accelerator 选择：
   - 优先 `GPU T4 x2`
   - 没有的话 `GPU T4` 也可以
4. Add Input：
   - Competition: **Predicting Electric Vehicle Purchases**
   - Dataset: **EV Adoption Behavior and Range Anxiety**
5. 上传/导入本 notebook，然后 **Run All**

如果只想先验证速度，可以把配置里的：

```python
N_FOLDS = 2
```

先跑两折。

正式生成 OOF 时恢复：

```python
N_FOLDS = 5
```

如果以后真的想做 10-fold，只需改成 `10`，但运行时间大约会翻倍。

In [1]:
# =========================
# 1. Imports & environment
# =========================

import os
import gc
import json
import math
import random
import warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
else:
    raise RuntimeError(
        "没有检测到 CUDA。请在 Kaggle Settings → Accelerator 中开启 GPU。"
    )

DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True

PyTorch: 2.14.0
CUDA available: False
CUDA device count: 0


RuntimeError: 没有检测到 CUDA。请在 Kaggle Settings → Accelerator 中开启 GPU。

In [ ]:
# =========================
# 2. Configuration
# =========================

TARGET = "Will_Buy_EV"
ID_COL = "id"

SEED = 42

# Public-style practical default.
N_FOLDS = 5
N_EPOCHS = 4

BATCH_SIZE = 4096
INFER_BATCH_SIZE = 16384
NUM_WORKERS = 2

HIDDEN_DIM = 384
N_BLOCKS = 3

LR = 2.0e-3
WEIGHT_DECAY = 1.0e-5

# Schedule visible in Vladimir Demidov's RealMLP PyTorch-style runs.
LABEL_SMOOTHING_SCHEDULE = [0.0341, 0.0200, 0.0059, 0.0000]
DROPOUT_SCHEDULE = [0.0184, 0.0068, 0.0025, 0.0009]

# Public S6E9 RealMLP notebook attaches the original EV dataset.
USE_ORIGINAL_DATA = True
ORIGINAL_REPEAT = 1

# Optional: use the best epoch of each fold by validation AUC.
SAVE_BEST_STATE = True

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "folds": N_FOLDS,
    "epochs": N_EPOCHS,
    "batch": BATCH_SIZE,
    "hidden": HIDDEN_DIM,
    "blocks": N_BLOCKS,
    "use_original": USE_ORIGINAL_DATA,
})

## 3. 自动寻找 competition / original data

Competition 文件通常位于：

```text
/kaggle/input/playground-series-s6e9/train.csv
/kaggle/input/playground-series-s6e9/test.csv
/kaggle/input/playground-series-s6e9/sample_submission.csv
```

原始数据文件：

```text
EV_Adoption_and_Range_Anxiety_Dataset.csv
```

代码会自动寻找，不依赖 dataset slug。

In [ ]:
# =========================
# 3. Load data
# =========================

def find_one(filename, preferred_text=None):
    root = Path("/kaggle/input")
    hits = [p for p in root.rglob(filename) if p.is_file()]
    if preferred_text:
        preferred = [p for p in hits if preferred_text.lower() in str(p).lower()]
        if preferred:
            hits = preferred
    if not hits:
        raise FileNotFoundError(
            f"找不到 {filename}。请确认已 Add Input。"
        )
    return sorted(hits)[0]


train_path = find_one("train.csv", "playground-series-s6e9")
test_path = find_one("test.csv", "playground-series-s6e9")
sample_path = find_one("sample_submission.csv", "playground-series-s6e9")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)

if TARGET not in train.columns:
    raise ValueError(
        f"train.csv 中找不到 {TARGET}. 实际列：{train.columns.tolist()}"
    )

y = train[TARGET].map({"No": 0, "Yes": 1}).astype(np.int8)

print("train:", train.shape)
print("test :", test.shape)
print("positive rate:", float(y.mean()))
print("target:", TARGET)

orig = None
if USE_ORIGINAL_DATA:
    try:
        orig_path = find_one(
            "EV_Adoption_and_Range_Anxiety_Dataset.csv"
        )
        orig = pd.read_csv(orig_path)
        print("original:", orig.shape, "|", orig_path)
    except FileNotFoundError:
        print(
            "WARNING: 没找到 original dataset；"
            "将自动使用 competition-only training。"
        )
        orig = None

## 4. Feature engineering

这里不使用 target encoding。

目的是让 NN 和 CatBoost/LightGBM/XGBoost 保持更不同的 feature view。

主要包含：

- 原始连续变量
- income / commute 的非线性变换
- charging interactions
- subsidy interactions
- income hinge
- digit / suffix categorical views
- 离散数值 categorical copy
- 少量类别交叉

所有特征都只依赖当前行，不依赖标签。

In [ ]:
# =========================
# 4. Feature engineering
# =========================

BASE_FEATURES = [
    "Age",
    "Gender",
    "Annual_Income_USD",
    "City_Type",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Current_Car_Type",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Home_Charging_Possible",
    "Environmental_Concern_Level",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]

def yes_no_numeric(s):
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").fillna(0).astype(np.float32)
    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "yes": 1.0, "no": 0.0,
            "true": 1.0, "false": 0.0,
            "1": 1.0, "0": 0.0,
        })
        .fillna(0.0)
        .astype(np.float32)
    )

def engineer_features(frame):
    missing = [c for c in BASE_FEATURES if c not in frame.columns]
    if missing:
        raise ValueError(f"缺少基础特征：{missing}")

    f = frame[BASE_FEATURES].copy()

    # ---------- numeric ----------
    num = pd.DataFrame(index=f.index)

    numeric_raw = [
        "Age",
        "Annual_Income_USD",
        "Daily_Commute_km",
        "Number_of_Cars_Owned",
        "Charging_Stations_Near_Home",
        "Charging_Stations_Near_Work",
        "Environmental_Concern_Level",
    ]

    for c in numeric_raw:
        num[c] = pd.to_numeric(f[c], errors="coerce").astype(np.float32)

    income = pd.to_numeric(f["Annual_Income_USD"], errors="coerce")
    commute = pd.to_numeric(f["Daily_Commute_km"], errors="coerce")
    age = pd.to_numeric(f["Age"], errors="coerce")
    cars = pd.to_numeric(f["Number_of_Cars_Owned"], errors="coerce")
    home = pd.to_numeric(f["Charging_Stations_Near_Home"], errors="coerce")
    work = pd.to_numeric(f["Charging_Stations_Near_Work"], errors="coerce")
    concern = pd.to_numeric(f["Environmental_Concern_Level"], errors="coerce")
    subsidy = yes_no_numeric(f["Subsidy_Available"])

    range_ord = (
        f["Range_Anxiety_Level"]
        .astype(str)
        .map({"Low": 0.0, "Medium": 1.0, "High": 2.0})
        .astype(np.float32)
    )

    num["income_log"] = np.log1p(income.clip(lower=0))
    num["income_sqrt"] = np.sqrt(income.clip(lower=0))
    num["commute_log"] = np.log1p(commute.clip(lower=0))
    num["commute_sq"] = commute ** 2
    num["age_sq"] = age ** 2

    num["charging_total"] = home + work
    num["charging_gap"] = home - work
    num["income_per_car"] = income / (cars + 1.0)
    num["income_per_commute"] = income / (commute + 1.0)

    num["subsidy_bin"] = subsidy
    num["range_ord"] = range_ord
    num["subsidy_x_income"] = subsidy * income
    num["subsidy_x_concern"] = subsidy * concern
    num["subsidy_x_home"] = subsidy * home
    num["subsidy_x_work"] = subsidy * work
    num["concern_x_range"] = concern * range_ord

    # Piecewise income.
    for knot in [30000, 31004, 42000, 60000, 80000, 100000, 120000, 150000, 170537]:
        num[f"income_hinge_{knot}"] = np.maximum(income - knot, 0.0)

    num["income_lt_31004"] = (income < 31004).astype(np.float32)
    num["income_31_42k"] = ((income >= 31004) & (income <= 41970)).astype(np.float32)
    num["income_ge_170537"] = (income >= 170537).astype(np.float32)
    num["commute_ge_83"] = (commute >= 83).astype(np.float32)

    # ---------- categorical ----------
    cat = pd.DataFrame(index=f.index)

    raw_cat = [
        "Gender",
        "City_Type",
        "Current_Car_Type",
        "Home_Charging_Possible",
        "Subsidy_Available",
        "Range_Anxiety_Level",
    ]
    for c in raw_cat:
        cat[c] = f[c].astype("string").fillna("__NA__")

    # Categorical copies of discrete numbers.
    discrete = [
        "Age",
        "Number_of_Cars_Owned",
        "Charging_Stations_Near_Home",
        "Charging_Stations_Near_Work",
        "Environmental_Concern_Level",
    ]
    for c in discrete:
        vals = pd.to_numeric(f[c], errors="coerce").fillna(-999)
        cat[f"{c}__cat"] = np.rint(vals).astype(np.int64).astype(str)

    income_fill = income.fillna(-999).round().astype(np.int64)
    commute_fill = commute.fillna(-999)

    cat["income_last_2"] = (income_fill % 100).astype(str)
    cat["income_last_3"] = (income_fill % 1000).astype(str)
    cat["income_thousand"] = (income_fill // 1000).astype(str)
    cat["commute_tenth"] = np.rint(commute_fill * 10).astype(np.int64).astype(str)
    cat["commute_integer"] = np.rint(commute_fill).astype(np.int64).astype(str)

    # Coarse buckets.
    cat["age_5yr"] = (pd.to_numeric(f["Age"], errors="coerce").fillna(-999) // 5).astype(np.int64).astype(str)
    cat["income_10k"] = (income_fill // 10000).astype(str)
    cat["commute_10km"] = (commute_fill // 10).astype(np.int64).astype(str)

    # Interactions.
    subsidy_str = cat["Subsidy_Available"].astype(str)
    range_str = cat["Range_Anxiety_Level"].astype(str)
    concern_str = cat["Environmental_Concern_Level__cat"].astype(str)

    cat["subsidy__range"] = subsidy_str + "__" + range_str
    cat["subsidy__concern"] = subsidy_str + "__" + concern_str
    cat["range__concern"] = range_str + "__" + concern_str
    cat["city__car"] = cat["City_Type"].astype(str) + "__" + cat["Current_Car_Type"].astype(str)
    cat["gender__city"] = cat["Gender"].astype(str) + "__" + cat["City_Type"].astype(str)
    cat["homecharge__subsidy"] = cat["Home_Charging_Possible"].astype(str) + "__" + subsidy_str

    return num, cat


train_num, train_cat = engineer_features(train)
test_num, test_cat = engineer_features(test)

print("numeric features:", train_num.shape[1])
print("categorical features:", train_cat.shape[1])
print("total engineered:", train_num.shape[1] + train_cat.shape[1])

## 5. Original data 预处理

原始文件的 ID 是 `Buyer_ID`，competition 是 `id`。

训练时只使用共同的 EV 特征与 `Will_Buy_EV`，不会把 `Buyer_ID` 当特征。

Original rows 会加入每一个 fold 的 training side，但**绝不会加入 validation side**。
所以 OOF 仍然只在 competition train rows 上评估。

In [ ]:
# =========================
# 5. Original dataset features
# =========================

orig_num = None
orig_cat = None
orig_y = None

if orig is not None:
    if TARGET not in orig.columns:
        print("WARNING: original dataset missing target; ignoring original data")
        orig = None
    else:
        orig_y = orig[TARGET].map({"No": 0, "Yes": 1})
        valid = orig_y.notna()

        orig_clean = orig.loc[valid].copy()
        orig_y = orig_y.loc[valid].astype(np.int8).reset_index(drop=True)

        orig_num, orig_cat = engineer_features(orig_clean)
        orig_num = orig_num.reset_index(drop=True)
        orig_cat = orig_cat.reset_index(drop=True)

        print("Original rows used:", len(orig_y))
        print("Original positive rate:", float(orig_y.mean()))
else:
    print("Competition-only mode")

## 6. Fold-local preprocessing

为了让 OOF 尽量诚实：

- numeric median / clipping / mean / std 只从该 fold 的 training side（+ optional original）计算；
- categorical vocabulary 只从 training side（+ optional original）建立；
- validation/test 未见类别映射为 `0`。

数值做：

1. median imputation
2. 0.5% / 99.5% winsorization
3. standardization
4. clip 到 `[-8, 8]`

In [ ]:
# =========================
# 6. Fold preprocessor
# =========================

@dataclass
class FoldPreprocessor:
    medians: np.ndarray
    lower: np.ndarray
    upper: np.ndarray
    means: np.ndarray
    stds: np.ndarray
    cat_maps: list
    cat_cardinalities: list


def fit_preprocessor(num_df, cat_df):
    x = num_df.to_numpy(dtype=np.float64)

    med = np.nanmedian(x, axis=0)
    x_imp = np.where(np.isnan(x), med, x)

    lo = np.nanquantile(x_imp, 0.005, axis=0)
    hi = np.nanquantile(x_imp, 0.995, axis=0)
    x_clip = np.clip(x_imp, lo, hi)

    means = x_clip.mean(axis=0)
    stds = x_clip.std(axis=0)
    stds = np.where(stds < 1e-6, 1.0, stds)

    maps = []
    cards = []

    for c in cat_df.columns:
        vals = cat_df[c].astype(str).fillna("__NA__")
        cats = pd.Index(pd.unique(vals))
        mapping = {v: i + 1 for i, v in enumerate(cats)}  # 0 = unknown
        maps.append(mapping)
        cards.append(len(mapping) + 1)

    return FoldPreprocessor(
        medians=med.astype(np.float32),
        lower=lo.astype(np.float32),
        upper=hi.astype(np.float32),
        means=means.astype(np.float32),
        stds=stds.astype(np.float32),
        cat_maps=maps,
        cat_cardinalities=cards,
    )


def transform_with_preprocessor(num_df, cat_df, prep):
    x = num_df.to_numpy(dtype=np.float32)
    x = np.where(np.isnan(x), prep.medians, x)
    x = np.clip(x, prep.lower, prep.upper)
    x = (x - prep.means) / prep.stds
    x = np.clip(x, -8.0, 8.0).astype(np.float32)

    cat_arrays = []
    for i, c in enumerate(cat_df.columns):
        mapped = (
            cat_df[c].astype(str)
            .map(prep.cat_maps[i])
            .fillna(0)
            .astype(np.int64)
            .to_numpy()
        )
        cat_arrays.append(mapped)

    cmat = np.column_stack(cat_arrays).astype(np.int64)

    return x, cmat

## 7. RealMLP-style PyTorch 模型

这是 GPU-native tabular MLP：

- categorical embeddings
- normalized engineered numerics
- learnable numeric projection
- residual MLP blocks
- SiLU activation
- epoch-wise label smoothing / dropout decay
- AdamW + OneCycleLR
- AMP

Kaggle T4×2 时自动 `DataParallel`。

In [ ]:
# =========================
# 7. Model
# =========================

def embedding_dim(cardinality):
    # Compact embedding rule; caps large exact-value categories.
    return int(min(32, max(4, round((cardinality ** 0.25) * 4))))


class TabDataset(Dataset):
    def __init__(self, x_num, x_cat, y=None):
        self.x_num = torch.from_numpy(x_num)
        self.x_cat = torch.from_numpy(x_cat)
        self.y = None if y is None else torch.from_numpy(np.asarray(y, dtype=np.float32))

    def __len__(self):
        return len(self.x_num)

    def __getitem__(self, idx):
        if self.y is None:
            return self.x_num[idx], self.x_cat[idx]
        return self.x_num[idx], self.x_cat[idx], self.y[idx]


class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim, dim * 2)
        self.fc2 = nn.Linear(dim * 2, dim)
        self.act = nn.SiLU()
        self.drop = nn.Dropout(dropout)

    def set_dropout(self, p):
        self.drop.p = float(p)

    def forward(self, x):
        h = self.norm(x)
        h = self.fc1(h)
        h = self.act(h)
        h = self.drop(h)
        h = self.fc2(h)
        h = self.drop(h)
        return x + h


class RealMLPStyle(nn.Module):
    def __init__(self, n_num, cat_cardinalities, hidden_dim=384, n_blocks=3, dropout=0.01):
        super().__init__()

        self.embeddings = nn.ModuleList()
        emb_dims = []

        for card in cat_cardinalities:
            d = embedding_dim(card)
            self.embeddings.append(nn.Embedding(card, d))
            emb_dims.append(d)

        cat_dim = sum(emb_dims)

        # Let numeric features enter both directly and through a learned representation.
        self.num_norm = nn.LayerNorm(n_num)
        self.num_proj = nn.Sequential(
            nn.Linear(n_num, max(64, n_num * 2)),
            nn.SiLU(),
        )
        num_proj_dim = max(64, n_num * 2)

        input_dim = n_num + num_proj_dim + cat_dim

        self.input = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
        )

        self.blocks = nn.ModuleList([
            ResidualBlock(hidden_dim, dropout)
            for _ in range(n_blocks)
        ])

        self.final_norm = nn.LayerNorm(hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def set_dropout(self, p):
        self.input[-1].p = float(p)
        for block in self.blocks:
            block.set_dropout(p)

    def forward(self, x_num, x_cat):
        num_n = self.num_norm(x_num)
        num_p = self.num_proj(num_n)

        cat_parts = [
            emb(x_cat[:, i])
            for i, emb in enumerate(self.embeddings)
        ]

        if cat_parts:
            x = torch.cat([num_n, num_p, *cat_parts], dim=1)
        else:
            x = torch.cat([num_n, num_p], dim=1)

        x = self.input(x)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)
        return self.head(x).squeeze(1)

## 8. 训练 / 推理函数

In [ ]:
# =========================
# 8. Train helpers
# =========================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    preds = []

    for batch in loader:
        x_num, x_cat = batch
        x_num = x_num.to(DEVICE, non_blocking=True)
        x_cat = x_cat.to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(x_num, x_cat)

        preds.append(torch.sigmoid(logits).float().cpu().numpy())

    return np.concatenate(preds)


def bce_with_label_smoothing(logits, targets, smoothing):
    if smoothing > 0:
        targets = targets * (1.0 - smoothing) + 0.5 * smoothing
    return nn.functional.binary_cross_entropy_with_logits(logits, targets)


def train_one_fold(
    fold,
    train_idx,
    valid_idx,
    full_num,
    full_cat,
    y,
    test_num_df,
    test_cat_df,
):
    set_seed(SEED + fold)

    # Fold training dataframe.
    tr_num = full_num.iloc[train_idx].reset_index(drop=True)
    tr_cat = full_cat.iloc[train_idx].reset_index(drop=True)
    tr_y = np.asarray(y)[train_idx].astype(np.float32)

    # Add original rows only to training side.
    if USE_ORIGINAL_DATA and orig_num is not None:
        extra_num = pd.concat([orig_num] * ORIGINAL_REPEAT, ignore_index=True)
        extra_cat = pd.concat([orig_cat] * ORIGINAL_REPEAT, ignore_index=True)
        extra_y = np.tile(np.asarray(orig_y, dtype=np.float32), ORIGINAL_REPEAT)

        fit_num = pd.concat([tr_num, extra_num], ignore_index=True)
        fit_cat = pd.concat([tr_cat, extra_cat], ignore_index=True)
        fit_y = np.concatenate([tr_y, extra_y])
    else:
        fit_num = tr_num
        fit_cat = tr_cat
        fit_y = tr_y

    va_num = full_num.iloc[valid_idx].reset_index(drop=True)
    va_cat = full_cat.iloc[valid_idx].reset_index(drop=True)
    va_y = np.asarray(y)[valid_idx].astype(np.float32)

    prep = fit_preprocessor(fit_num, fit_cat)

    xtr_num, xtr_cat = transform_with_preprocessor(fit_num, fit_cat, prep)
    xva_num, xva_cat = transform_with_preprocessor(va_num, va_cat, prep)
    xte_num, xte_cat = transform_with_preprocessor(test_num_df, test_cat_df, prep)

    train_ds = TabDataset(xtr_num, xtr_cat, fit_y)
    valid_ds = TabDataset(xva_num, xva_cat)
    test_ds = TabDataset(xte_num, xte_cat)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
        persistent_workers=(NUM_WORKERS > 0),
    )

    valid_loader = DataLoader(
        valid_ds,
        batch_size=INFER_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=INFER_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )

    model = RealMLPStyle(
        n_num=xtr_num.shape[1],
        cat_cardinalities=prep.cat_cardinalities,
        hidden_dim=HIDDEN_DIM,
        n_blocks=N_BLOCKS,
        dropout=DROPOUT_SCHEDULE[0],
    ).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters())

    if torch.cuda.device_count() > 1:
        print(f"Using DataParallel on {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    total_steps = max(1, len(train_loader) * N_EPOCHS)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=LR,
        total_steps=total_steps,
        pct_start=0.15,
        anneal_strategy="cos",
        div_factor=10.0,
        final_div_factor=50.0,
    )

    scaler = torch.amp.GradScaler("cuda")

    best_auc = -np.inf
    best_state = None
    best_epoch = None
    epoch_records = []

    for epoch in range(N_EPOCHS):
        model.train()

        dropout = DROPOUT_SCHEDULE[min(epoch, len(DROPOUT_SCHEDULE) - 1)]
        smoothing = LABEL_SMOOTHING_SCHEDULE[
            min(epoch, len(LABEL_SMOOTHING_SCHEDULE) - 1)
        ]

        target_model = model.module if isinstance(model, nn.DataParallel) else model
        target_model.set_dropout(dropout)

        losses = []

        for x_num_b, x_cat_b, y_b in train_loader:
            x_num_b = x_num_b.to(DEVICE, non_blocking=True)
            x_cat_b = x_cat_b.to(DEVICE, non_blocking=True)
            y_b = y_b.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(x_num_b, x_cat_b)
                loss = bce_with_label_smoothing(
                    logits,
                    y_b,
                    smoothing=smoothing,
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            losses.append(float(loss.detach().cpu()))

        va_pred = predict_loader(model, valid_loader)
        va_auc = float(roc_auc_score(va_y, va_pred))

        epoch_records.append({
            "fold": fold,
            "epoch": epoch + 1,
            "train_loss": float(np.mean(losses)),
            "valid_auc": va_auc,
            "label_smoothing": smoothing,
            "dropout": dropout,
        })

        improved = va_auc > best_auc

        print(
            f"epoch {epoch+1}/{N_EPOCHS} "
            f"auc={va_auc:.6f} "
            f"best={max(best_auc, va_auc):.6f} "
            f"loss={np.mean(losses):.5f} "
            f"ls={smoothing:.4f} "
            f"drop={dropout:.4f}"
            + (" ✓" if improved else "")
        )

        if improved:
            best_auc = va_auc
            best_epoch = epoch + 1

            if SAVE_BEST_STATE:
                raw_model = model.module if isinstance(model, nn.DataParallel) else model
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in raw_model.state_dict().items()
                }

    if SAVE_BEST_STATE and best_state is not None:
        raw_model = model.module if isinstance(model, nn.DataParallel) else model
        raw_model.load_state_dict(best_state)

    valid_pred = predict_loader(model, valid_loader)
    test_pred = predict_loader(model, test_loader)

    final_auc = float(roc_auc_score(va_y, valid_pred))

    print(
        f"Fold {fold} | best epoch={best_epoch} "
        f"| AUC={final_auc:.9f} | params={n_params:,}"
    )

    del (
        model, optimizer, scheduler, scaler,
        train_loader, valid_loader, test_loader,
        train_ds, valid_ds, test_ds,
        xtr_num, xtr_cat, xva_num, xva_cat, xte_num, xte_cat,
    )
    gc.collect()
    torch.cuda.empty_cache()

    return valid_pred, test_pred, final_auc, epoch_records

## 9. K-Fold training

默认 5-fold。

每一折完成后都会立刻保存 partial prediction 到 `/kaggle/working`，方便检查。

In [ ]:
# =========================
# 9. Cross-validation
# =========================

folds = list(
    StratifiedKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=SEED,
    ).split(train, y)
)

oof = np.full(len(train), np.nan, dtype=np.float64)
coverage = np.zeros(len(train), dtype=np.int8)
test_sum = np.zeros(len(test), dtype=np.float64)

fold_scores = []
all_epoch_records = []

for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
    print()
    print("#" * 70)
    print(f"### Fold {fold}/{N_FOLDS}")
    print("#" * 70)

    va_pred, te_pred, fold_auc, epoch_records = train_one_fold(
        fold=fold,
        train_idx=tr_idx,
        valid_idx=va_idx,
        full_num=train_num,
        full_cat=train_cat,
        y=y,
        test_num_df=test_num,
        test_cat_df=test_cat,
    )

    oof[va_idx] = va_pred
    coverage[va_idx] += 1
    test_sum += te_pred

    fold_scores.append({
        "fold": fold,
        "auc": fold_auc,
        "n_train_competition": len(tr_idx),
        "n_valid": len(va_idx),
    })

    all_epoch_records.extend(epoch_records)

    # Partial saves after each completed fold.
    pd.DataFrame(fold_scores).to_csv(
        OUT_DIR / "fold_scores_realmlp_v91.csv",
        index=False,
    )

    pd.DataFrame(all_epoch_records).to_csv(
        OUT_DIR / "epoch_scores_realmlp_v91.csv",
        index=False,
    )

    np.save(
        OUT_DIR / f"realmlp_v91_oof_partial_after_fold{fold}.npy",
        oof,
    )
    np.save(
        OUT_DIR / f"realmlp_v91_test_sum_after_fold{fold}.npy",
        test_sum,
    )

assert np.all(coverage == 1), "OOF coverage error"
assert np.isfinite(oof).all(), "OOF has NaN"

test_pred = test_sum / N_FOLDS

overall_auc = float(roc_auc_score(y, oof))

print()
print("=" * 70)
print("Fold AUCs:")
print(
    pd.DataFrame(fold_scores)
    .to_string(index=False, float_format=lambda v: f"{v:.9f}")
)
print()
print(f"Overall RealMLP V9.1 OOF AUC = {overall_auc:.9f}")
print("=" * 70)

## 10. 输出 OOF + submission

这是之后拿回 Mac 做：

```text
RealMLP vs V8 Spearman
→ V8 + RealMLP Nested Hill
```

所需要的文件。

In [ ]:
# =========================
# 10. Save final outputs
# =========================

oof_df = pd.DataFrame({
    "id": train["id"].to_numpy(),
    "target": y.to_numpy(),
    "prediction": oof,
})

submission = sample[["id"]].merge(
    pd.DataFrame({
        "id": test["id"].to_numpy(),
        TARGET: test_pred,
    }),
    on="id",
    how="left",
    validate="one_to_one",
    sort=False,
)

assert submission[TARGET].notna().all()
assert np.isfinite(submission[TARGET]).all()

oof_path = OUT_DIR / "oof_realmlp_v91.csv"
sub_path = OUT_DIR / "submission_realmlp_v91.csv"

oof_df.to_csv(oof_path, index=False)
submission.to_csv(sub_path, index=False)

report = {
    "model": "RealMLP-style PyTorch V9.1",
    "target": TARGET,
    "folds": N_FOLDS,
    "epochs": N_EPOCHS,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "hidden_dim": HIDDEN_DIM,
    "n_blocks": N_BLOCKS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "use_original_data": bool(USE_ORIGINAL_DATA and orig is not None),
    "original_rows": int(len(orig_y)) if orig_y is not None else 0,
    "gpu_count": int(torch.cuda.device_count()),
    "gpu_names": [
        torch.cuda.get_device_name(i)
        for i in range(torch.cuda.device_count())
    ],
    "fold_scores": fold_scores,
    "overall_oof_auc": overall_auc,
    "numeric_features": train_num.columns.tolist(),
    "categorical_features": train_cat.columns.tolist(),
}

(OUT_DIR / "realmlp_v91_report.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print("Saved:")
print(oof_path)
print(sub_path)
print(OUT_DIR / "fold_scores_realmlp_v91.csv")
print(OUT_DIR / "epoch_scores_realmlp_v91.csv")
print(OUT_DIR / "realmlp_v91_report.json")

## 11. 结果怎么判断

跑完后先看：

```text
Overall RealMLP V9.1 OOF AUC = ...
```

不要只看某一个 fold。

### 大概判断

如果：

```text
RealMLP OOF >= 0.9460
```

已经值得拿回 V8 做相关性分析。

如果它单模只有：

```text
0.9457 ~ 0.9460
```

也先别判死刑——关键要看它与 V8 的 **Spearman**。

神经网络的主要价值是提供与 GBDT 不同的排序。

---

### Kaggle 跑完以后下载这两个

```text
oof_realmlp_v91.csv
submission_realmlp_v91.csv
```

然后把它们放到你 Mac 的：

```text
/Users/yw/unsw/Kaggle/outputs/v9_realmlp_gpu/
```

接下来我们本地几秒钟就能做：

```text
RealMLP vs V8 correlation
+
V8 / RealMLP nested hill
+
最终 submission
```

## 12. 如果 GPU OOM

把：

```python
BATCH_SIZE = 4096
```

改成：

```python
BATCH_SIZE = 2048
```

如果还不够：

```python
HIDDEN_DIM = 320
```

正常 T4 / T4×2 不应该需要降得很低。

## 如果跑得太久

先做速度测试：

```python
N_FOLDS = 2
```

确定每折只需要几分钟后，再恢复 5 folds 正式跑。

不要再用 Mac CPU 跑这版。